<a href="https://colab.research.google.com/github/EconShark/23_big_data_analysis_class/blob/main/Baitap03/Machine_Learning_with_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Clone only the repository metadata
!git clone --depth 1 --filter=blob:none --sparse https://github.com/EconShark/23_big_data_analysis_class.git

# 2. Check out only the Baitap03/data directory
!cd 23_big_data_analysis_class && git sparse-checkout set Baitap03/data

# 3. Copy the datasets to /content/data
!mkdir -p /content/data
!cp -rn 23_big_data_analysis_class/Baitap03/data/. /content/data/

# 4. Remove the cloned repository
!rm -rf 23_big_data_analysis_class

Cloning into '23_big_data_analysis_class'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 30 (delta 0), reused 22 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 6.35 KiB | 6.35 MiB/s, done.
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 4 (delta 0), reused 4 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), 168.31 KiB | 1.98 MiB/s, done.
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 0), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), 3.57 MiB | 8.13 MiB/s, done.
Updating files: 100% (4/4), done.
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead


<a id="setup"></a>
## Khởi tạo SparkSession

In [2]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    print("Đang cài PySpark...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
else:
    print("PySpark đã sẵn sàng.")

PySpark đã sẵn sàng.


In [3]:
from pathlib import Path

candidates = [
    Path.cwd() / "data",
    Path("/content/data"),
    Path.cwd().parent / "data"
]
DATA_DIR = next((p for p in candidates if p.exists()), None)

print("DATA_DIR =", DATA_DIR)
if DATA_DIR:
    for p in sorted(DATA_DIR.iterdir()):
        print(" -", p.name)

DATA_DIR = /content/data
 - flights-larger.csv
 - sms.csv


In [4]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("MachineLearningWithPySpark")
    .getOrCreate()
)

print("PySpark version:", pyspark.__version__)
print("Spark version  :", spark.version)
print("Master         :", spark.sparkContext.master)

PySpark version: 4.0.4
Spark version  : 4.0.4
Master         : local[*]


<a id="loading"></a>
## 1. Đọc và khảo sát dữ liệu

- `flights-larger.csv`: bài toán dự đoán chuyến bay trễ và thời gian bay.
- `sms.csv`: bài toán phân loại SMS spam.

### 1.1. Dữ liệu chuyến bay

In [5]:
flights = spark.read.csv(
    str(DATA_DIR / "flights-larger.csv"),
    header=True,
    inferSchema=True,
    nullValue="NA"
)

flights.printSchema()
flights.show(5)
print("Số bản ghi:", flights.count())

root
 |-- mon: integer (nullable = true)
 |-- dom: integer (nullable = true)
 |-- dow: integer (nullable = true)
 |-- carrier: string (nullable = true)
 |-- flight: integer (nullable = true)
 |-- org: string (nullable = true)
 |-- mile: integer (nullable = true)
 |-- depart: double (nullable = true)
 |-- duration: integer (nullable = true)
 |-- delay: integer (nullable = true)

+---+---+---+-------+------+---+----+------+--------+-----+
|mon|dom|dow|carrier|flight|org|mile|depart|duration|delay|
+---+---+---+-------+------+---+----+------+--------+-----+
| 10| 10|  1|     OO|  5836|ORD| 157|  8.18|      51|   27|
|  1|  4|  1|     OO|  5866|ORD| 466|  15.5|     102| NULL|
| 11| 22|  1|     OO|  6016|ORD| 738|  7.17|     127|  -19|
|  2| 14|  5|     B6|   199|JFK|2248| 21.17|     365|   60|
|  5| 25|  3|     WN|  1675|SJC| 386| 12.92|      85|   22|
+---+---+---+-------+------+---+----+------+--------+-----+
only showing top 5 rows
Số bản ghi: 275000


### 1.2. Các cột chính

- `mon`: tháng
- `dom`: ngày trong tháng
- `dow`: ngày trong tuần
- `carrier`: hãng bay
- `flight`: số hiệu chuyến bay
- `org`: sân bay xuất phát
- `mile`: quãng đường
- `depart`: giờ khởi hành
- `duration`: thời gian bay
- `delay`: độ trễ

### 1.3. Dữ liệu SMS với schema tường minh

Định nghĩa `StructType`.

In [7]:
sms_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("text", StringType(), True),
    StructField("label", IntegerType(), True)
])

sms = spark.read.csv(
    str(DATA_DIR / "sms.csv"),
    sep=";",
    header=False,
    schema=sms_schema
)

sms.show(5, truncate=False)
sms.printSchema()

+---+---------------------------------------------------------------------------------------------------------------+-----+
|id |text                                                                                                           |label|
+---+---------------------------------------------------------------------------------------------------------------+-----+
|1  |Sorry, I'll call later in meeting                                                                              |0    |
|2  |Dont worry. I guess he's busy.                                                                                 |0    |
|3  |Call FREEPHONE 0800 542 0578 now!                                                                              |1    |
|4  |Win a 1000 cash prize or a prize worth 5000                                                                    |1    |
|5  |Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...|0    |
+---+---

<a id="prep"></a>
## 2. Chuẩn bị dữ liệu cho Machine Learning

Trong Machine Learning, dữ liệu thô thường chưa thể đưa trực tiếp vào mô hình.

Các bước thường gặp:

1. loại cột không cần thiết;
2. xử lý missing values;
3. chuyển đổi đơn vị;
4. tạo label;
5. mã hóa biến phân loại;
6. ghép các predictor thành `features`.

### 2.1. Kiểm tra missing values

In [8]:
flights.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in flights.columns
]).show()

+---+---+---+-------+------+---+----+------+--------+-----+
|mon|dom|dow|carrier|flight|org|mile|depart|duration|delay|
+---+---+---+-------+------+---+----+------+--------+-----+
|  0|  0|  0|      0|     0|  0|   0|     0|       0|16711|
+---+---+---+-------+------+---+----+------+--------+-----+



### 2.2. Làm sạch và tạo label
`withColumn()` được dùng để tạo feature mới.

Ở đây:

- mile → km;
- tạo `label = 1` nếu delay >= 15 phút.

In [10]:
flights_ml = (
    flights
    .drop("flight")
    .filter(F.col("delay").isNotNull())
    .dropna()
    .withColumn("km", F.round(F.col("mile") * 1.60934, 0))
    .drop("mile")
    .withColumn("label", (F.col("delay") >= 15).cast("integer"))
)

flights_ml.select(
    "mon", "carrier", "org", "km", "duration", "delay", "label"
).show(10)

+---+-------+---+------+--------+-----+-----+
|mon|carrier|org|    km|duration|delay|label|
+---+-------+---+------+--------+-----+-----+
| 10|     OO|ORD| 253.0|      51|   27|    1|
| 11|     OO|ORD|1188.0|     127|  -19|    0|
|  2|     B6|JFK|3618.0|     365|   60|    1|
|  5|     WN|SJC| 621.0|      85|   22|    1|
|  3|     B6|LGA|1732.0|     182|   70|    1|
|  5|     B6|ORD|1191.0|     130|   47|    1|
|  1|     UA|SFO|1093.0|     123|  135|    1|
|  8|     US|LGA| 344.0|      71|  -10|    0|
|  5|     AA|ORD|1926.0|     195|  -11|    0|
|  8|     B6|JFK|1902.0|     198|   20|    1|
+---+-------+---+------+--------+-----+-----+
only showing top 10 rows


<a id="categorical"></a>
## 3. StringIndexer, OneHotEncoder và VectorAssembler

Machine Learning algorithms thường cần dữ liệu số.

### 3.1. StringIndexer

`StringIndexer` biến chuỗi category thành chỉ số số.

Ví dụ:

```text
AA -> 0
OO -> 1
B6 -> 2
...
```

Thứ tự mặc định thường phụ thuộc tần suất xuất hiện.

In [14]:
from pyspark.ml.feature import StringIndexer

carrier_indexer = StringIndexer(
    inputCol="carrier",
    outputCol="carrier_idx",
    handleInvalid="keep"
)

flights_idx = carrier_indexer.fit(flights_ml).transform(flights_ml)

# Khởi tạo StringIndexer cho cột 'org'
org_indexer = StringIndexer(
    inputCol="org",
    outputCol="org_idx",
    handleInvalid="keep"
)

# Khớp và chuyển đổi trên flights_idx để thêm cột org_idx
flights_idx = org_indexer.fit(flights_idx).transform(flights_idx)

### 3.2. OneHotEncoder

One-Hot Encoding tạo vector nhị phân/sparse vector, tránh tạo quan hệ thứ tự giả giữa các category.

In [12]:
from pyspark.ml.feature import OneHotEncoder

onehot = OneHotEncoder(
    inputCols=["org_idx"],
    outputCols=["org_dummy"],
    dropLast=True
)

<a id="assembler"></a>
### 3.3. VectorAssembler

Spark ML yêu cầu predictor được gom trong một cột vector tên thường là **`features`**.

`VectorAssembler` thực hiện việc này.

In [15]:
from pyspark.ml.feature import VectorAssembler

# Khởi tạo VectorAssembler để gộp các đặc trưng
assembler = VectorAssembler(
    inputCols=[
        "mon", "dom", "dow",
        "carrier_idx", "org_idx",
        "km", "depart", "duration"
    ],
    outputCol="features"
)

# Chuyển đổi dữ liệu sẵn sàng cho mô hình
flights_ready = assembler.transform(flights_idx)

flights_ready.select(
    "features", "label", "delay"
).show(5, truncate=False)

+-----------------------------------------+-----+-----+
|features                                 |label|delay|
+-----------------------------------------+-----+-----+
|[10.0,10.0,1.0,2.0,0.0,253.0,8.18,51.0]  |1    |27   |
|[11.0,22.0,1.0,2.0,0.0,1188.0,7.17,127.0]|0    |-19  |
|[2.0,14.0,5.0,4.0,2.0,3618.0,21.17,365.0]|1    |60   |
|[5.0,25.0,3.0,3.0,5.0,621.0,12.92,85.0]  |1    |22   |
|[3.0,28.0,1.0,4.0,3.0,1732.0,13.33,182.0]|1    |70   |
+-----------------------------------------+-----+-----+
only showing top 5 rows


### 3.4. Chia train/test

In [16]:
train_df, test_df = flights_ready.randomSplit(
    [0.8, 0.2],
    seed=23
)

print("Train:", train_df.count())
print("Test :", test_df.count())
print("Train ratio:", train_df.count() / flights_ready.count())

Train: 206212
Test : 52077
Train ratio: 0.7983770117968632


<a id="dt"></a>
## 4. Decision Tree Classification

Decision Tree phân chia dữ liệu qua các nút:

```text
Root
 ├── nhánh trái
 └── nhánh phải
```

Quá trình tiếp tục cho đến khi tạo các leaf node.

Decision Tree được dùng để phân loại xe theo nơi sản xuất.  
Ở phần mở rộng, ta dùng nó để dự đoán chuyến bay có trễ >= 15 phút hay không.

In [17]:
from pyspark.ml.classification import DecisionTreeClassifier

tree = DecisionTreeClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=5,
    seed=23
)

tree_model = tree.fit(train_df)
tree_pred = tree_model.transform(test_df)

tree_pred.select(
    "label", "prediction", "probability"
).show(10, truncate=False)

+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|1    |0.0       |[0.6225186017127615,0.37748139828723853]|
|1    |1.0       |[0.3195589266923886,0.6804410733076114] |
|0    |1.0       |[0.3195589266923886,0.6804410733076114] |
|1    |0.0       |[0.6225186017127615,0.37748139828723853]|
|1    |1.0       |[0.3195589266923886,0.6804410733076114] |
|0    |1.0       |[0.3195589266923886,0.6804410733076114] |
|1    |1.0       |[0.3195589266923886,0.6804410733076114] |
|1    |0.0       |[0.6225186017127615,0.37748139828723853]|
|0    |0.0       |[0.6225186017127615,0.37748139828723853]|
|0    |0.0       |[0.6225186017127615,0.37748139828723853]|
+-----+----------+----------------------------------------+
only showing top 10 rows


### 4.1. Xem cấu trúc cây

In [18]:
print(tree_model.toDebugString[:5000])

DecisionTreeClassificationModel: uid=DecisionTreeClassifier_b203aef5b506, depth=5, numNodes=43, numClasses=2, numFeatures=8
  If (feature 6 <= 10.96)
   If (feature 4 in {4.0,5.0,6.0,7.0})
    If (feature 7 <= 251.5)
     Predict: 0.0
    Else (feature 7 > 251.5)
     If (feature 3 in {1.0,4.0,6.0})
      If (feature 4 in {5.0})
       Predict: 0.0
      Else (feature 4 not in {5.0})
       Predict: 1.0
     Else (feature 3 not in {1.0,4.0,6.0})
      If (feature 0 <= 8.5)
       Predict: 0.0
      Else (feature 0 > 8.5)
       Predict: 1.0
   Else (feature 4 not in {4.0,5.0,6.0,7.0})
    If (feature 6 <= 9.49)
     Predict: 0.0
    Else (feature 6 > 9.49)
     If (feature 0 <= 5.5)
      If (feature 5 <= 1050.5)
       Predict: 0.0
      Else (feature 5 > 1050.5)
       Predict: 1.0
     Else (feature 0 > 5.5)
      If (feature 0 <= 10.5)
       Predict: 0.0
      Else (feature 0 > 10.5)
       Predict: 1.0
  Else (feature 6 > 10.96)
   If (feature 4 in {4.0,5.0,6.0,7.0})
    If (feat

<a id="logistic"></a>
## 5. Logistic Regression và đánh giá classifier

Logistic Regression dùng hàm logistic để biến linear score thành xác suất trong khoảng 0–1.

Mặc định:

- xác suất >= ngưỡng → class 1;
- xác suất < ngưỡng → class 0.

PySpark:

In [19]:
from pyspark.ml.classification import LogisticRegression

logistic = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    maxIter=50
)

log_model = logistic.fit(train_df)
log_pred = log_model.transform(test_df)

log_pred.select(
    "label", "prediction", "probability"
).show(10, truncate=False)

+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|1    |0.0       |[0.557245791644765,0.44275420835523505] |
|1    |1.0       |[0.2646308554040308,0.7353691445959691] |
|0    |1.0       |[0.2438573328127501,0.7561426671872499] |
|1    |0.0       |[0.5405863866128278,0.45941361338717224]|
|1    |1.0       |[0.29158128138704686,0.7084187186129531]|
|0    |1.0       |[0.31373727084730363,0.6862627291526964]|
|1    |1.0       |[0.3180490430367354,0.6819509569632646] |
|1    |1.0       |[0.4241927240351956,0.5758072759648044] |
|0    |0.0       |[0.5356616493670768,0.46433835063292317]|
|0    |1.0       |[0.49707685136788693,0.5029231486321131]|
+-----+----------+----------------------------------------+
only showing top 10 rows


<a id="class_eval"></a>
### 5.1. Confusion Matrix

Ta cần bốn thành phần:

- TP: dự đoán 1, thực tế 1.
- FP: dự đoán 1, thực tế 0.
- FN: dự đoán 0, thực tế 1.
- TN: dự đoán 0, thực tế 0.

In [21]:
log_pred.groupBy("label", "prediction").count().show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       0.0| 9675|
|    0|       0.0|14754|
|    1|       1.0|16798|
|    0|       1.0|10850|
+-----+----------+-----+



### 5.2. Accuracy, Precision, Recall, F1, AUC

In [22]:
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator
)

multi_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction"
)

binary_eval = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction"
)

for metric in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
    value = multi_eval.evaluate(
        log_pred,
        {multi_eval.metricName: metric}
    )
    print(f"{metric:18s}: {value:.4f}")

auc = binary_eval.evaluate(log_pred)
print(f"{'AUC':18s}: {auc:.4f}")

accuracy          : 0.6059
f1                : 0.6055
weightedPrecision : 0.6058
weightedRecall    : 0.6059
AUC               : 0.6456


<a id="text"></a>
## 6. Text Mining và SMS Spam Classification

Slide trình bày pipeline:

```text
Text
 ↓
Remove punctuation
 ↓
Tokenizer
 ↓
StopWordsRemover
 ↓
HashingTF
 ↓
IDF
 ↓
features
```

Đây là cách biến văn bản thành vector số cho Machine Learning.

### 6.1. Làm sạch và tokenize

In [26]:
from pyspark.sql.functions import regexp_replace

sms_clean = sms.withColumn(
    "text",
    regexp_replace("text", r"[_():;,.!?\-]", " ")
)

sms_clean = sms_clean.withColumn(
    "text",
    regexp_replace("text", r"[0-9]", " ")
)

sms_clean = sms_clean.withColumn(
    "text",
    regexp_replace("text", r" +", " ")
)

sms_clean.show(5, truncate=False)

+---+-------------------------------------------------------------------------------------------------------+-----+
|id |text                                                                                                   |label|
+---+-------------------------------------------------------------------------------------------------------+-----+
|1  |Sorry I'll call later in meeting                                                                       |0    |
|2  |Dont worry I guess he's busy                                                                           |0    |
|3  |Call FREEPHONE now                                                                                     |1    |
|4  |Win a cash prize or a prize worth                                                                      |1    |
|5  |Go until jurong point crazy Available only in bugis n great world la e buffet Cine there got amore wat |0    |
+---+-------------------------------------------------------------------

#### Tokenizer

In [27]:
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(
    inputCol="text",
    outputCol="tokens"
)

sms_tokens = tokenizer.transform(sms_clean)

sms_tokens.select("text", "tokens").show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------+
|text                                                                                                   |tokens                                                                                                                     |
+-------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------+
|Sorry I'll call later in meeting                                                                       |[sorry, i'll, call, later, in, meeting]                                                                                    |
|Dont worry I guess he's busy                                                   

#### StopWordsRemover

In [28]:
from pyspark.ml.feature import StopWordsRemover

remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="words"
)

sms_words = remover.transform(sms_tokens)

sms_words.select("tokens", "words").show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------+
|tokens                                                                                                                     |words                                                                                              |
+---------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------+
|[sorry, i'll, call, later, in, meeting]                                                                                    |[sorry, call, later, meeting]                                                                      |
|[dont, worry, i, guess, he's, busy]                                                            

### 6.2. HashingTF + IDF

In [29]:
from pyspark.ml.feature import HashingTF

hasher = HashingTF(
    inputCol="words",
    outputCol="hash",
    numFeatures=1024
)

sms_hash = hasher.transform(sms_words)

sms_hash.select("id", "words", "hash").show(5, truncate=False)

+---+---------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------+
|id |words                                                                                              |hash                                                                                                                                     |
+---+---------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------+
|1  |[sorry, call, later, meeting]                                                                      |(1024,[138,384,577,996],[1.0,1.0,1.0,1.0])                                                                                               |
|2  |[dont, worry, guess

In [30]:
from pyspark.ml.feature import IDF

idf = IDF(
    inputCol="hash",
    outputCol="features"
)

idf_model = idf.fit(sms_hash)
sms_tfidf = idf_model.transform(sms_hash)

sms_tfidf.select(
    "id", "label", "features"
).show(5, truncate=False)

+---+-----+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|id |label|features                                                                                                                                                                                                                                                                                                                                                                        |
+---+-----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

<a id="sms"></a>
### 6.3. Logistic Regression cho SMS

Bài toán:

```text
Input : SMS text
Output: 0 = normal, 1 = spam
```

Ta dùng Logistic Regression trên TF-IDF features.

In [31]:
sms_model_df = sms_tfidf.select("label", "features")

sms_train, sms_test = sms_model_df.randomSplit(
    [0.8, 0.2],
    seed=13
)

sms_logistic = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    regParam=0.2
)

sms_log_model = sms_logistic.fit(sms_train)
sms_pred = sms_log_model.transform(sms_test)

sms_pred.groupBy("label", "prediction").count().show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       0.0|   39|
|    0|       0.0|  932|
|    1|       1.0|  121|
|    0|       1.0|    4|
+-----+----------+-----+



In [32]:
sms_auc = BinaryClassificationEvaluator(
    labelCol="label"
).evaluate(sms_pred)

sms_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    metricName="f1"
).evaluate(sms_pred)

print("SMS AUC:", sms_auc)
print("SMS F1 :", sms_f1)

SMS AUC: 0.9912793803418793
SMS F1 : 0.9587173795339076


<a id="regression"></a>
## 7. Linear Regression

Linear Regression để dự đoán fuel consumption.  
Phần mở rộng dùng dataset chuyến bay để dự đoán **duration**.

Ta thử mô hình đơn giản:

```text
duration ~ km
```

In [33]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

reg_base = (
    flights
    .filter(F.col("duration").isNotNull())
    .filter(F.col("mile").isNotNull())
    .withColumn("km", F.round(F.col("mile") * 1.60934, 0))
)

reg_assembler = VectorAssembler(
    inputCols=["km"],
    outputCol="features"
)

reg_data = reg_assembler.transform(reg_base)

reg_train, reg_test = reg_data.randomSplit([0.8, 0.2], seed=42)

In [34]:
lr_model = lr = LinearRegression(
    labelCol="duration",
    featuresCol="features"
).fit(reg_train)


reg_pred = lr_model.transform(reg_test)

reg_pred.select(
    "km", "duration", "prediction"
).show(10)

+------+--------+------------------+
|    km|duration|        prediction|
+------+--------+------------------+
|4001.0|     300| 347.3336415989867|
|3983.0|     375| 345.9700268069072|
|1395.0|     135|149.91252225681146|
|1926.0|     180|190.13915862315645|
|1180.0|     115| 133.6249011291953|
|1180.0|     130| 133.6249011291953|
|1180.0|     130| 133.6249011291953|
|1091.0|     110| 126.8825835461356|
|2356.0|     205|222.71440087838872|
|2659.0|     251|245.66858321172683|
+------+--------+------------------+
only showing top 10 rows


### 7.1. Các metric regression

In [37]:
for metric in ["rmse", "mae", "mse", "r2"]:
    evaluator = RegressionEvaluator(
        labelCol="duration",
        predictionCol="prediction",
        metricName=metric
    )
    print(f"{metric:4s}: {evaluator.evaluate(reg_pred):.4f}")

rmse: 17.1476
mae : 13.1121
mse : 294.0405
r2  : 0.9613


### 7.2. Intercept và coefficient

In [38]:
print("Intercept   :", lr_model.intercept)
print("Coefficients:", lr_model.coefficients)

Intercept   : 44.23237587065089
Coefficients: [0.07575637733774951]


<a id="engineering"></a>
## 8. Feature Engineering và Bucketing

- bucketing;
- log, sqrt, power;
- product;
- ratio.

Bucketing biến biến liên tục thành nhóm rời rạc.

### 8.1. Bucketizer cho giờ khởi hành

In [39]:
from pyspark.ml.feature import Bucketizer

# Dữ liệu depart có dạng giờ thập phân khoảng 0–24
bucketizer = Bucketizer(
    splits=[0, 6, 12, 18, 24],
    inputCol="depart",
    outputCol="depart_bucket",
    handleInvalid="keep"
)

bucketed = bucketizer.transform(
    flights.filter(F.col("depart").isNotNull())
)

bucketed.select(
    "depart", "depart_bucket"
).show(10)

bucketed.groupBy("depart_bucket").count().orderBy("depart_bucket").show()

+------+-------------+
|depart|depart_bucket|
+------+-------------+
|  8.18|          1.0|
|  15.5|          2.0|
|  7.17|          1.0|
| 21.17|          3.0|
| 12.92|          2.0|
| 13.33|          2.0|
|  9.58|          1.0|
| 12.75|          2.0|
|  13.0|          2.0|
| 14.42|          2.0|
+------+-------------+
only showing top 10 rows
+-------------+------+
|depart_bucket| count|
+-------------+------+
|          0.0|   911|
|          1.0| 98550|
|          2.0|104471|
|          3.0| 71068|
+-------------+------+



### 8.2. Feature công thức

In [40]:
engineered = (
    flights
    .filter(F.col("mile").isNotNull())
    .filter(F.col("duration").isNotNull())
    .withColumn("km", F.col("mile") * 1.60934)
    .withColumn("speed_km_per_min", F.col("km") / F.col("duration"))
    .withColumn("sqrt_km", F.sqrt("km"))
    .withColumn("log_km", F.log1p("km"))
)

engineered.select(
    "km", "duration", "speed_km_per_min", "sqrt_km", "log_km"
).show(10)

+----------+--------+-----------------+------------------+------------------+
|        km|duration| speed_km_per_min|           sqrt_km|            log_km|
+----------+--------+-----------------+------------------+------------------+
| 252.66638|      51|4.954242745098039| 15.89548300618764| 5.536019939100953|
| 749.95244|     102|7.352474901960784|27.385259538664226| 6.621342320863969|
|1187.69292|     127|9.351912755905511| 34.46292094411035|  7.08060959588236|
|3617.79632|     365|9.911770739726027| 60.14811983761421| 8.193896741239387|
| 621.20524|      85| 7.30829694117647|24.923989247309507|6.4332700061574215|
|1731.64984|     182| 9.51455956043956| 41.61309697679326| 7.457407215058698|
| 1190.9116|     130| 9.16085846153846|34.509587073739375| 7.083313683801401|
|1092.74186|     123|8.884080162601627| 33.05664623037249|6.9973599953584165|
| 344.39876|      71| 4.85068676056338|18.557983726687553| 5.844699575665601|
|1926.37998|     195|9.878871692307692| 43.89054545115611|  7.56

<a id="regularization"></a>
## 9. Regularization: Ridge, Lasso, Elastic Net

Linear Regression chuẩn tối thiểu hóa lỗi.  
Regularization bổ sung penalty lên coefficient.

### Ridge

- `elasticNetParam = 0`
- penalty L2.

### Lasso

- `elasticNetParam = 1`
- penalty L1.
- có thể đẩy một số coefficient về 0.

### Elastic Net

- `0 < elasticNetParam < 1`
- kết hợp L1 và L2.

In [42]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd

# Tái tạo lại DataFrame 'reg_full' bị thiếu do ô trước bị xóa
reg_idx = StringIndexer(
    inputCol="org",
    outputCol="org_idx",
    handleInvalid="keep"
).fit(reg_base).transform(reg_base)

reg_ohe = OneHotEncoder(
    inputCols=["org_idx"],
    outputCols=["org_dummy"]
).fit(reg_idx).transform(reg_idx)

reg_assembler2 = VectorAssembler(
    inputCols=["km", "org_dummy"],
    outputCol="features2"
)

reg_full = reg_assembler2.transform(reg_ohe)

# Thực hiện chia tập dữ liệu train/test
reg_train3, reg_test3 = reg_full.randomSplit([0.8, 0.2], seed=23)

models_reg = []

for name, alpha in [
    ("Ridge", 0.0),
    ("ElasticNet", 0.5),
    ("Lasso", 1.0)
]:
    model = LinearRegression(
        labelCol="duration",
        featuresCol="features2",
        regParam=0.1,
        elasticNetParam=alpha
    ).fit(reg_train3)

    pred = model.transform(reg_test3)
    rmse = RegressionEvaluator(
        labelCol="duration",
        predictionCol="prediction",
        metricName="rmse"
    ).evaluate(pred)

    models_reg.append({
        "model": name,
        "rmse": rmse,
        "num_coefficients": len(model.coefficients),
        "zero_coefficients": sum(abs(x) < 1e-10 for x in model.coefficients)
    })

pd.DataFrame(models_reg)

,model,rmse,num_coefficients,zero_coefficients
0,Ridge,11.289664,9,0
1,ElasticNet,11.291365,9,1
2,Lasso,11.294537,9,0


<a id="pipeline"></a>
## 10. Pipeline và Data Leakage

```text
.fit()       -> chỉ dùng TRAIN
.transform() -> dùng cho TRAIN và TEST
```

Nếu fit bộ tiền xử lý trên toàn bộ dataset trước khi chia train/test, test data đã tác động vào preprocessing → **data leakage**.

`Pipeline` giúp đóng gói các stage đúng thứ tự.

### Pipeline Regression (Minh họa)

In [43]:
from pyspark.ml import Pipeline

pipeline_data = (
    flights
    .filter(F.col("duration").isNotNull())
    .filter(F.col("mile").isNotNull())
    .filter(F.col("org").isNotNull())
    .withColumn("km", F.round(F.col("mile") * 1.60934, 0))
)

pipe_train, pipe_test = pipeline_data.randomSplit(
    [0.8, 0.2],
    seed=13
)

pipe_indexer = StringIndexer(
    inputCol="org",
    outputCol="org_idx",
    handleInvalid="keep"
)

pipe_onehot = OneHotEncoder(
    inputCols=["org_idx"],
    outputCols=["org_dummy"]
)

pipe_assembler = VectorAssembler(
    inputCols=["km", "org_dummy"],
    outputCol="features"
)

pipe_lr = LinearRegression(
    labelCol="duration",
    featuresCol="features"
)

pipeline = Pipeline(
    stages=[
        pipe_indexer,
        pipe_onehot,
        pipe_assembler,
        pipe_lr
    ]
)

pipeline_model = pipeline.fit(pipe_train)
pipe_pred = pipeline_model.transform(pipe_test)

pipe_rmse = RegressionEvaluator(
    labelCol="duration",
    metricName="rmse"
).evaluate(pipe_pred)

print("Pipeline RMSE:", pipe_rmse)

Pipeline RMSE: 11.187971350851406


<a id="cv"></a>
## 11. Cross Validation và Grid Search

Train/test split chỉ cho một phép chia dữ liệu.

Cross Validation:

1. chia training data thành nhiều fold;
2. lần lượt dùng mỗi fold làm validation;
3. tính metric trung bình;
4. chọn model/hyperparameter tốt nhất.

Test set vẫn được giữ riêng để đánh giá cuối cùng.

In [45]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

cv_regression = LinearRegression(
    labelCol="duration",
    featuresCol="features"
)

cv_pipeline = Pipeline(
    stages=[
        pipe_indexer,
        pipe_onehot,
        pipe_assembler,
        cv_regression
    ]
)

param_grid = (
    ParamGridBuilder()
    .addGrid(cv_regression.regParam, [0.01, 0.1, 1.0])
    .addGrid(cv_regression.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)

print("Số cấu hình:", len(param_grid))

cv_evaluator = RegressionEvaluator(
    labelCol="duration",
    metricName="rmse"
)

cv = CrossValidator(
    estimator=cv_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=cv_evaluator,
    numFolds=3,
    seed=13,
    parallelism=2
)

Số cấu hình: 9


In [46]:
# Huấn luyện cross-validation
# Có thể mất vài phút tùy máy.
cv_model = cv.fit(pipe_train)

best_predictions = cv_model.transform(pipe_test)
best_rmse = cv_evaluator.evaluate(best_predictions)

print("Best test RMSE:", best_rmse)
print("Average CV metrics:")
print(cv_model.avgMetrics)

Best test RMSE: 11.18794629815799
Average CV metrics:
[np.float64(11.21222312526637), np.float64(11.212220771736682), np.float64(11.212250742825184), np.float64(11.212645439450151), np.float64(11.213404714714505), np.float64(11.215621374066805), np.float64(11.253953198119447), np.float64(11.329378009751911), np.float64(11.546458968429102)]


### 11.1. Xem mô hình tốt nhất

In [47]:
best_pipeline = cv_model.bestModel
best_lr = best_pipeline.stages[-1]

print("regParam       =", best_lr.getRegParam())
print("elasticNetParam=", best_lr.getElasticNetParam())

regParam       = 0.01
elasticNetParam= 0.5


<a id="ensemble"></a>
## 12. Ensemble Models

**Ensemble**: kết hợp nhiều cây/mô hình.

Các ví dụ trong code ZIP:

- Decision Tree;
- Gradient-Boosted Trees (GBT);
- Random Forest;
- Cross Validation cho Random Forest.

In [49]:
ensemble_data = (
    flights
    .filter(F.col("delay").isNotNull())
    .filter(F.col("duration").isNotNull())
    .filter(F.col("depart").isNotNull())
    .filter(F.col("mon").isNotNull())
    .withColumn("label", (F.col("delay") >= 15).cast("integer"))
)

ensemble_assembler = VectorAssembler(
    inputCols=["mon", "depart", "duration"],
    outputCol="features"
)

ensemble_data = ensemble_assembler.transform(ensemble_data)

ens_train, ens_test = ensemble_data.randomSplit(
    [0.8, 0.2],
    seed=17
)

from pyspark.ml.classification import (
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier
)

models = {
    "Decision Tree": DecisionTreeClassifier(seed=17),
    "Random Forest": RandomForestClassifier(numTrees=30, seed=17),
    "GBT": GBTClassifier(maxIter=20, seed=17)
}

### 12.1. So sánh mô hình (Decision Tree, RF, GBT)

In [50]:
ensemble_results = []

for name, estimator in models.items():
    model = estimator.fit(ens_train)
    pred = model.transform(ens_test)

    auc = BinaryClassificationEvaluator(
        labelCol="label"
    ).evaluate(pred)

    f1 = MulticlassClassificationEvaluator(
        labelCol="label",
        metricName="f1"
    ).evaluate(pred)

    ensemble_results.append({
        "model": name,
        "AUC": auc,
        "F1": f1
    })

pd.DataFrame(ensemble_results).sort_values("AUC", ascending=False)

,model,AUC,F1
2,GBT,0.679002,0.628652
1,Random Forest,0.666685,0.620376
0,Decision Tree,0.618192,0.620846


<a id="project"></a>
## 4. BÀI TẬP MỞ RỘNG

### Project A - Flight Delay Classification

#### Input

- month, day-of-month, day-of-week;
- carrier;
- origin airport;
- distance;
- departure time;
- duration.

#### Output

```text
label = 1 nếu delay >= 15 phút
label = 0 nếu ngược lại
```

#### Yêu cầu

1. Làm sạch dữ liệu.
2. Tạo label: delay >= 15 phút.
3. Mã hóa carrier và org.
4. Tạo Pipeline preprocessing.
5. So sánh:
   - Logistic Regression;
   - Decision Tree;
   - Random Forest;
   - GBT.
5. Báo cáo Accuracy, F1, AUC.
6. Tuning ít nhất một mô hình bằng Cross Validation.
7. Giải thích vì sao chọn mô hình cuối cùng.

#### 1. Preprocessing and Feature Engineering Pipeline for Project A

In [51]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

# 1. Data Cleaning & Feature Extraction
cleaned_flights_proj_a = (
    flights
    .drop("flight")
    .filter(F.col("delay").isNotNull())
    .dropna()
    .withColumn("km", F.round(F.col("mile") * 1.60934, 0))
    .drop("mile")
    .withColumn("label", (F.col("delay") >= 15).cast("integer"))
)

# 2. Train/Test Split (80/20)
train_proj_a, test_proj_a = cleaned_flights_proj_a.randomSplit([0.8, 0.2], seed=42)

# 3. Create Pipeline stages
carrier_indexer = StringIndexer(inputCol="carrier", outputCol="carrier_idx", handleInvalid="keep")
org_indexer = StringIndexer(inputCol="org", outputCol="org_idx", handleInvalid="keep")
org_encoder = OneHotEncoder(inputCols=["org_idx"], outputCols=["org_dummy"])

assembler = VectorAssembler(
    inputCols=["mon", "dom", "dow", "carrier_idx", "org_dummy", "km", "depart", "duration"],
    outputCol="features"
)

# Assemble stages into the preprocessing pipeline
preprocessing_pipeline = Pipeline(stages=[carrier_indexer, org_indexer, org_encoder, assembler])

#### 2. Multi-model Training & Evaluation

In [52]:
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import pandas as pd

# Define dictionary of models to compare
classifiers = {
    "Logistic Regression": LogisticRegression(featuresCol="features", labelCol="label", maxIter=30),
    "Decision Tree": DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=5, seed=42),
    "Random Forest": RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=30, seed=42),
    "Gradient-Boosted Trees (GBT)": GBTClassifier(featuresCol="features", labelCol="label", maxIter=20, seed=42)
}

comparison_metrics = []

for name, clf in classifiers.items():
    # Combine preprocessing stages with the current classifier
    full_pipeline = Pipeline(stages=[preprocessing_pipeline, clf])

    # Train
    model = full_pipeline.fit(train_proj_a)

    # Predict
    predictions = model.transform(test_proj_a)

    # Evaluate metrics
    auc = BinaryClassificationEvaluator(labelCol="label").evaluate(predictions)
    f1 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1").evaluate(predictions)
    acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy").evaluate(predictions)

    comparison_metrics.append({
        "Model": name,
        "Accuracy": acc,
        "F1-Score": f1,
        "AUC": auc
    })

# Display summary DataFrame
pd.DataFrame(comparison_metrics).sort_values(by="AUC", ascending=False)

,Model,Accuracy,F1-Score,AUC
3,Gradient-Boosted Trees (GBT),0.665294,0.664974,0.727418
2,Random Forest,0.631501,0.631095,0.683178
0,Logistic Regression,0.610892,0.610380,0.653452
1,Decision Tree,0.631733,0.631709,0.598907


#### 3. Tuning the Best Model (GBT) using Cross Validation

In [53]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# Instantiate GBT
gbt = GBTClassifier(featuresCol="features", labelCol="label", seed=42)

# Combine preprocessing pipeline with the GBT estimator
gbt_pipeline = Pipeline(stages=[preprocessing_pipeline, gbt])

# Create a hyperparameter grid
param_grid_gbt = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth, [3, 5])
    .addGrid(gbt.maxIter, [10, 20])
    .build()
)

print(f"Number of configurations to test: {len(param_grid_gbt)}")

# Set up Binary Classification Evaluator (optimizing for AUC)
evaluator_auc = BinaryClassificationEvaluator(labelCol="label")

# Initialize 3-Fold Cross Validator
cv_gbt = CrossValidator(
    estimator=gbt_pipeline,
    estimatorParamMaps=param_grid_gbt,
    evaluator=evaluator_auc,
    numFolds=3,
    seed=42,
    parallelism=2
)

# Train the CrossValidator model
cv_gbt_model = cv_gbt.fit(train_proj_a)

Number of configurations to test: 4


In [54]:
# Evaluate the best model on test data
tuned_predictions = cv_gbt_model.transform(test_proj_a)

tuned_auc = evaluator_auc.evaluate(tuned_predictions)
tuned_acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy").evaluate(tuned_predictions)
tuned_f1 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1").evaluate(tuned_predictions)

print("=== Tuned GBT Test Set Performance ===")
print(f"Accuracy : {tuned_acc:.4f}")
print(f"F1-Score : {tuned_f1:.4f}")
print(f"AUC      : {tuned_auc:.4f}")

# Retrieve best model parameters
best_gbt_model = cv_gbt_model.bestModel.stages[-1]
print("\n=== Best Parameters ===")
print(f"maxDepth : {best_gbt_model.getMaxDepth()}")
print(f"maxIter  : {best_gbt_model.getMaxIter()}")

=== Tuned GBT Test Set Performance ===
Accuracy : 0.6653
F1-Score : 0.6650
AUC      : 0.7274

=== Best Parameters ===
maxDepth : 5
maxIter  : 20


#### 4. Final Model Selection Explanation

Lý do lựa chọn GBT (Gradient-Boosted Trees):

1\. Hiệu quả dự báo cao nhất: GBT đạt AUC cao nhất (\~0,727), vượt Random Forest (\~0,683), Logistic Regression (\~0,653) và Decision Tree (\~0,598), cho thấy khả năng phân biệt chuyến bay trễ và không trễ tốt hơn.

2\. Cơ chế tăng cường: GBT xây dựng các cây tuần tự, trong đó mỗi cây mới tập trung khắc phục sai số của các cây trước, giúp nâng cao khả năng dự báo đối với dữ liệu phức tạp.

3\. Tối ưu hóa mô hình: Sử dụng kiểm định chéo 3 lần (3-Fold Cross-Validation) để tinh chỉnh `maxDepth` và `maxIter`, nhằm cân bằng hiệu quả dự báo và hạn chế hiện tượng quá khớp (overfitting).

### Project B - SMS Spam Classification

#### Input

Nội dung SMS.

#### Output

- 0: normal;
- 1: spam.

#### Pipeline

```text
Text
 → Tokenizer
 → StopWordsRemover
 → HashingTF
 → IDF
 → LogisticRegression
```

1. Làm sạch text và tokenize.
2. Loại stop words.
3. HashingTF + IDF.
4. Logistic Regression.
5. Thử numFeatures = 1024, 4096, 16384.
6. Thử regParam/elasticNetParam.
7. So sánh F1 và AUC.

#### 1. Preprocessing and Text Mining Pipeline for Project B

In [55]:
from pyspark.sql.functions import regexp_replace, lower
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# 1. Clean the SMS text
sms_clean_proj_b = (
    sms
    .filter(F.col("text").isNotNull())
    .withColumn("text_clean", regexp_replace("text", r"[_():;,.!?\-]", " "))
    .withColumn("text_clean", regexp_replace("text_clean", r"[0-9]", " "))
    .withColumn("text_clean", regexp_replace("text_clean", r" +", " "))
    .withColumn("text_clean", lower(F.col("text_clean")))
)

# 2. Split train/test (80/20)
train_proj_b, test_proj_b = sms_clean_proj_b.randomSplit([0.8, 0.2], seed=42)

# 3. Define Pipeline NLP stages (excluding HashingTF and Model for parameter tuning)
tokenizer = Tokenizer(inputCol="text_clean", outputCol="tokens")
remover = StopWordsRemover(inputCol="tokens", outputCol="words")

# Base pipeline for feature preparation up to text tokens extraction
nlp_prep_pipeline = Pipeline(stages=[tokenizer, remover])

#### 2. Multi-parameter Grid Search and Evaluation for Project B

In [56]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import pandas as pd

# We will manually iterate over combinations of numFeatures, regParam, and elasticNetParam
# to find the best performing configuration on our test set.

num_features_list = [1024, 4096, 16384]
reg_params = [0.01, 0.1, 0.2]
elastic_net_params = [0.0, 0.5, 1.0]

results_proj_b = []

# Fits base pipeline stages first to speed up the loop
fitted_nlp_prep = nlp_prep_pipeline.fit(train_proj_b)
train_prepped = fitted_nlp_prep.transform(train_proj_b)
test_prepped = fitted_nlp_prep.transform(test_proj_b)

for num_f in num_features_list:
    hasher = HashingTF(inputCol="words", outputCol="hash", numFeatures=num_f)
    idf = IDF(inputCol="hash", outputCol="features")

    # Fit TF-IDF transformation
    train_tfidf = idf.fit(hasher.transform(train_prepped)).transform(hasher.transform(train_prepped))
    test_tfidf = idf.fit(hasher.transform(train_prepped)).transform(hasher.transform(test_prepped))

    for reg in reg_params:
        for l1_ratio in elastic_net_params:
            lr = LogisticRegression(
                featuresCol="features",
                labelCol="label",
                regParam=reg,
                elasticNetParam=l1_ratio
            )

            # Fit model
            lr_model = lr.fit(train_tfidf)
            predictions = lr_model.transform(test_tfidf)

            # Evaluate metrics
            auc = BinaryClassificationEvaluator(labelCol="label").evaluate(predictions)
            f1 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1").evaluate(predictions)

            results_proj_b.append({
                "numFeatures": num_f,
                "regParam": reg,
                "elasticNetParam": l1_ratio,
                "F1-Score": f1,
                "AUC": auc
            })

# Organize and display results ranked by F1-Score
pdf_results = pd.DataFrame(results_proj_b).sort_values(by="F1-Score", ascending=False)
pdf_results.head(10)

,numFeatures,regParam,elasticNetParam,F1-Score,AUC
9,4096,0.01,0.0,0.973833,0.976417
0,1024,0.01,0.0,0.973005,0.969746
18,16384,0.01,0.0,0.968933,0.980564
12,4096,0.10,0.0,0.966380,0.980852
21,16384,0.10,0.0,0.965326,0.984149
19,16384,0.01,0.5,0.964798,0.971596
3,1024,0.10,0.0,0.962425,0.976501
15,4096,0.20,0.0,0.961061,0.981678
10,4096,0.01,0.5,0.960597,0.975166
24,16384,0.20,0.0,0.959984,0.984998


**Tổng kết phân tích Dự án B**

Dựa trên kết quả tìm kiếm lưới (Grid Search), các cấu hình tốt nhất là:

| Số đặc trưng (`numFeatures`) | `regParam` | `elasticNetParam` | F1-Score | AUC    |
| ---------------------------- | ---------- | ----------------- | -------- | ------ |
| 4.096                        | 0,01       | 0,0 (L2 – Ridge)  | 0,9738   | 0,9764 |
| 1.024                        | 0,01       | 0,0 (L2 – Ridge)  | 0,9730   | 0,9697 |
| 16.384                       | 0,01       | 0,0 (L2 – Ridge)  | 0,9689   | 0,9806 |

**Những phát hiện chính và cơ sở lựa chọn**

1\. Kích thước từ vựng tối ưu

Cấu hình 4.096 đặc trưng đạt F1-Score cao nhất (0,9738), đồng thời có AUC cao (0,9764), thể hiện sự cân bằng tốt giữa khả năng phân loại và phân biệt thư rác. Kích thước 1.024 có thể làm tăng va chạm băm (hash collisions), trong khi 16.384 đạt AUC cao nhất nhưng có F1-Score thấp hơn. Vì vậy, 4.096 là lựa chọn hợp lý nếu ưu tiên F1-Score.

2\. Hiệu quả của chuẩn hóa L2 (Ridge)

Các cấu hình hàng đầu đều sử dụng `elasticNetParam = 0.0`, tương ứng với chuẩn hóa L2. Phương pháp này giúp hạn chế các hệ số mô hình quá lớn mà không loại bỏ hoàn toàn đặc trưng, phù hợp với bài toán phân loại thư rác vốn dựa trên nhiều tín hiệu ngôn ngữ khác nhau như từ khóa, URL và cách diễn đạt.

Lưu ý: Kết quả hiện tại chưa đủ để khẳng định 4.096 là kích thước tối ưu tuyệt đối hoặc 16.384 gây quá khớp. Cần đánh giá thêm trên tập kiểm thử độc lập để xác nhận khả năng khái quát hóa của mô hình.

<a id="exercise"></a>
## 5. BÀI TẬP THÊM

### Bài 1 - Data Preparation

Với `flights-larger.csv`:

1. đếm missing value ở từng cột;
2. loại `flight`;
3. loại dòng thiếu `delay`;
4. tạo `km`;
5. tạo `label`.

---

### Bài 2 - StringIndexer và OneHotEncoder

1. index `carrier`;
2. index `org`;
3. one-hot encode `org`;
4. giải thích tại sao không nên dùng index trực tiếp như một biến liên tục.

---

### Bài 3 - Decision Tree

1. huấn luyện cây với `maxDepth = 3, 5, 10`;
2. so sánh Accuracy/F1/AUC;
3. in `toDebugString`.

---

### Bài 4 - Logistic Regression

1. huấn luyện Logistic Regression;
2. tính confusion matrix;
3. tính Precision, Recall, F1;
4. tính AUC.

---

### Bài 5 - SMS

1. loại punctuation;
2. tokenize;
3. remove stop words;
4. HashingTF;
5. IDF;
6. Logistic Regression;
7. tính F1 và AUC.

---

### Bài 6 - Regression

Dự đoán `duration`.

1. model chỉ dùng `km`;
2. model dùng `km + org`;
3. so sánh RMSE;
4. giải thích coefficient.

---

### Bài 7 - Regularization

Huấn luyện:

- Linear Regression;
- Ridge;
- Lasso;
- Elastic Net.

So sánh:

- RMSE;
- số coefficient gần 0.

---

### Bài 8 - Pipeline

Xây dựng Pipeline:

```text
StringIndexer
 → OneHotEncoder
 → VectorAssembler
 → LinearRegression
```

Giải thích tại sao phải fit pipeline trên training data.

---

### Bài 9 - Cross Validation

Grid:

```text
regParam = [0.01, 0.1, 1.0]
elasticNetParam = [0, 0.5, 1]
```

1. có bao nhiêu model configuration?
2. chạy 3-fold CV;
3. tìm model tốt nhất;
4. đánh giá trên test set.

---

### Bài 10 - Ensemble

So sánh:

- Decision Tree;
- Random Forest;
- GBT.

Chọn mô hình dựa trên AUC và F1.

## Solutions to Section 5: BÀI TẬP THÊM

### Bài 1 & 2 - Data Preparation, StringIndexer, and OneHotEncoder

* **Why shouldn't we use category index values directly as continuous variables?**
  Using index values directly (like 0, 1, 2 for carrier/origin codes) forces the machine learning model to assume an implicit ordinal relationship (e.g., that carrier indexed as 2 is "greater than" or twice as large as carrier 1). This mathematical ordering is arbitrary and false, leading to model degradation. Transforming index categories to One-Hot Encoded sparse vectors avoids this issue.

In [57]:
# --- Bài 1 ---
# 1. Count missing values per column
missing_counts = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in flights.columns]
print("=== Missing values per column ===")
flights.select(missing_counts).show()

# 2-5. Drop flight, drop null delay, convert to km, and build binary delay label
flights_prepared = (
    flights
    .drop("flight")
    .filter(F.col("delay").isNotNull())
    .dropna()
    .withColumn("km", F.round(F.col("mile") * 1.60934, 0))
    .drop("mile")
    .withColumn("label", (F.col("delay") >= 15).cast("integer"))
)
print(f"Prepared dataset records count: {flights_prepared.count()}")

# --- Bài 2 ---
# Indexing and encoding
carrier_indexer_b2 = StringIndexer(inputCol="carrier", outputCol="carrier_idx", handleInvalid="keep")
org_indexer_b2 = StringIndexer(inputCol="org", outputCol="org_idx", handleInvalid="keep")
org_encoder_b2 = OneHotEncoder(inputCols=["org_idx"], outputCols=["org_dummy"])

assembler_b2 = VectorAssembler(
    inputCols=["mon", "dom", "dow", "carrier_idx", "org_dummy", "km", "depart", "duration"],
    outputCol="features"
)

# Prepare data ready
pipeline_b2 = Pipeline(stages=[carrier_indexer_b2, org_indexer_b2, org_encoder_b2, assembler_b2])
flights_ready_b2 = pipeline_b2.fit(flights_prepared).transform(flights_prepared)
flights_ready_b2.select("features", "label").show(3, truncate=False)

=== Missing values per column ===
+---+---+---+-------+------+---+----+------+--------+-----+
|mon|dom|dow|carrier|flight|org|mile|depart|duration|delay|
+---+---+---+-------+------+---+----+------+--------+-----+
|  0|  0|  0|      0|     0|  0|   0|     0|       0|16711|
+---+---+---+-------+------+---+----+------+--------+-----+

Prepared dataset records count: 258289
+-------------------------------------------------------------------+-----+
|features                                                           |label|
+-------------------------------------------------------------------+-----+
|(15,[0,1,2,3,4,12,13,14],[10.0,10.0,1.0,2.0,1.0,253.0,8.18,51.0])  |1    |
|(15,[0,1,2,3,4,12,13,14],[11.0,22.0,1.0,2.0,1.0,1188.0,7.17,127.0])|0    |
|(15,[0,1,2,3,6,12,13,14],[2.0,14.0,5.0,4.0,1.0,3618.0,21.17,365.0])|1    |
+-------------------------------------------------------------------+-----+
only showing top 3 rows


### Bài 3 - Decision Tree Depth Tuning

In [58]:
# Split data
train_b3, test_b3 = flights_ready_b2.randomSplit([0.8, 0.2], seed=42)

depths = [3, 5, 10]
results_b3 = []

for depth in depths:
    dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=depth, seed=42)
    dt_model = dt.fit(train_b3)
    preds = dt_model.transform(test_b3)

    # Evaluate metrics
    acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy").evaluate(preds)
    f1 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1").evaluate(preds)
    auc = BinaryClassificationEvaluator(labelCol="label").evaluate(preds)

    results_b3.append({
        "maxDepth": depth,
        "Accuracy": acc,
        "F1-Score": f1,
        "AUC": auc
    })

    print(f"\n=== Decision Tree Structure (maxDepth = {depth}) ===")
    print(dt_model.toDebugString[:500] + "\n[Truncated...]\n")

pd.DataFrame(results_b3)


=== Decision Tree Structure (maxDepth = 3) ===
DecisionTreeClassificationModel: uid=DecisionTreeClassifier_e4713f2a30fd, depth=3, numNodes=13, numClasses=2, numFeatures=15
  If (feature 13 <= 11.01)
   If (feature 3 in {3.0,7.0,8.0})
    If (feature 12 <= 3538.0)
     Predict: 0.0
    Else (feature 12 > 3538.0)
     Predict: 1.0
   Else (feature 3 not in {3.0,7.0,8.0})
    Predict: 0.0
  Else (feature 13 > 11.01)
   If (feature 0 <= 7.5)
    If (feature 3 in {3.0,5.0,6.0,7.0,8.0})
     Predict: 0.0
    Else (feature 3 not in {3.0,5.0,6.0,7.0
[Truncated...]


=== Decision Tree Structure (maxDepth = 5) ===
DecisionTreeClassificationModel: uid=DecisionTreeClassifier_08dfa155b192, depth=5, numNodes=41, numClasses=2, numFeatures=15
  If (feature 13 <= 11.01)
   If (feature 3 in {3.0,7.0,8.0})
    If (feature 12 <= 3538.0)
     If (feature 13 <= 9.524999999999999)
      Predict: 0.0
     Else (feature 13 > 9.524999999999999)
      If (feature 5 in {1.0})
       Predict: 1.0
      Else (feat

,maxDepth,Accuracy,F1-Score,AUC
0,3,0.621254,0.619027,0.599361
1,5,0.631733,0.631709,0.598907
2,10,0.663032,0.663028,0.591627


### Bài 4 - Logistic Regression Classification Metrics

In [60]:
# Train Logistic Regression classifier
lr_b4 = LogisticRegression(featuresCol="features", labelCol="label", maxIter=30)
lr_model_b4 = lr_b4.fit(train_b3)
preds_b4 = lr_model_b4.transform(test_b3)

# 1. Confusion matrix
print("=== Confusion Matrix ===")
preds_b4.groupBy("label", "prediction").count().show()

# Evaluate Precision, Recall, F1, and AUC
precision = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedPrecision").evaluate(preds_b4)
recall = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedRecall").evaluate(preds_b4)
f1_b4 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1").evaluate(preds_b4)
auc_b4 = BinaryClassificationEvaluator(labelCol="label").evaluate(preds_b4)

print("=== Test Set Evaluation Metrics ===")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1_b4:.4f}")
print(f"AUC       : {auc_b4:.4f}")

=== Confusion Matrix ===
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       0.0| 9141|
|    0|       0.0|14847|
|    1|       1.0|16752|
|    0|       1.0|10986|
+-----+----------+-----+

=== Test Set Evaluation Metrics ===
Precision : 0.6114
Recall    : 0.6109
F1-Score  : 0.6104
AUC       : 0.6535


### Bài 5 - SMS Spam Classification Pipeline and Evaluation

In [61]:
# 1. Clean the SMS text data
sms_clean_b5 = (
    sms
    .filter(F.col("text").isNotNull())
    .withColumn("text_clean", regexp_replace("text", r"[_():;,.!?\-]", " "))
    .withColumn("text_clean", regexp_replace("text_clean", r"[0-9]", " "))
    .withColumn("text_clean", regexp_replace("text_clean", r" +", " "))
    .withColumn("text_clean", lower(F.col("text_clean")))
)

# 2. Tokenize and Remove Stop Words
tok_b5 = Tokenizer(inputCol="text_clean", outputCol="tokens")
rem_b5 = StopWordsRemover(inputCol="tokens", outputCol="words")

# 3. Apply HashingTF & IDF
hash_b5 = HashingTF(inputCol="words", outputCol="hash_features", numFeatures=1024)
idf_b5 = IDF(inputCol="hash_features", outputCol="features")

# 4. Logistic Regression Classifier
lr_b5 = LogisticRegression(featuresCol="features", labelCol="label", regParam=0.2)

# Construct the full text mining and classification Pipeline
sms_pipeline = Pipeline(stages=[tok_b5, rem_b5, hash_b5, idf_b5, lr_b5])

# Split training and testing sets (80/20)
sms_train_b5, sms_test_b5 = sms_clean_b5.randomSplit([0.8, 0.2], seed=42)

# Train and evaluate
sms_model_b5 = sms_pipeline.fit(sms_train_b5)
sms_preds = sms_model_b5.transform(sms_test_b5)

f1_b5 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1").evaluate(sms_preds)
auc_b5 = BinaryClassificationEvaluator(labelCol="label").evaluate(sms_preds)

print("=== SMS Spam Classification Evaluation ===")
print(f"F1-Score : {f1_b5:.4f}")
print(f"AUC      : {auc_b5:.4f}")

=== SMS Spam Classification Evaluation ===
F1-Score : 0.9525
AUC      : 0.9772


### Bài 6 & 7 - Regression Comparison and Regularization Analysis

In [62]:
# Prepare regression baseline dataset
reg_base_b6 = (
    flights
    .filter(F.col("duration").isNotNull())
    .filter(F.col("mile").isNotNull())
    .filter(F.col("org").isNotNull())
    .withColumn("km", F.round(F.col("mile") * 1.60934, 0))
)

# Model 1: dur ~ km
assembler_m1 = VectorAssembler(inputCols=["km"], outputCol="features_m1")
data_m1 = assembler_m1.transform(reg_base_b6)
train_m1, test_m1 = data_m1.randomSplit([0.8, 0.2], seed=23)

lr_m1 = LinearRegression(featuresCol="features_m1", labelCol="duration")
model_m1 = lr_m1.fit(train_m1)
preds_m1 = model_m1.transform(test_m1)
rmse_m1 = RegressionEvaluator(labelCol="duration", metricName="rmse").evaluate(preds_m1)

# Model 2: dur ~ km + org
org_idx_m2 = StringIndexer(inputCol="org", outputCol="org_idx", handleInvalid="keep")
org_ohe_m2 = OneHotEncoder(inputCols=["org_idx"], outputCols=["org_dummy"])
assembler_m2 = VectorAssembler(inputCols=["km", "org_dummy"], outputCol="features_m2")

prep_pipe_m2 = Pipeline(stages=[org_idx_m2, org_ohe_m2, assembler_m2])
prepped_data_m2 = prep_pipe_m2.fit(reg_base_b6).transform(reg_base_b6)
train_m2, test_m2 = prepped_data_m2.randomSplit([0.8, 0.2], seed=23)

lr_m2 = LinearRegression(featuresCol="features_m2", labelCol="duration")
model_m2 = lr_m2.fit(train_m2)
preds_m2 = model_m2.transform(test_m2)
rmse_m2 = RegressionEvaluator(labelCol="duration", metricName="rmse").evaluate(preds_m2)

print("=== Model Performance Comparison ===")
print(f"Model 1 (km only) RMSE        : {rmse_m1:.4f}")
print(f"Model 2 (km + org OHE) RMSE   : {rmse_m2:.4f}")
print(f"Model 1 intercept & slope     : {model_m1.intercept:.4f} & {model_m1.coefficients[0]:.4f}")

# --- Bài 7: Regularization comparison ---
reg_types = [("Ridge", 0.0), ("ElasticNet", 0.5), ("Lasso", 1.0)]
results_b7 = []

for name, elastic_ratio in reg_types:
    reg_lr = LinearRegression(featuresCol="features_m2", labelCol="duration", regParam=0.1, elasticNetParam=elastic_ratio)
    reg_model = reg_lr.fit(train_m2)
    reg_preds = reg_model.transform(test_m2)

    rmse_val = RegressionEvaluator(labelCol="duration", metricName="rmse").evaluate(reg_preds)
    near_zeros = sum(1 for w in reg_model.coefficients if abs(w) < 1e-5)

    results_b7.append({
        "Regularization": name,
        "RMSE": rmse_val,
        "Total Coefficients": len(reg_model.coefficients),
        "Near-Zero Coefficients": near_zeros
    })

pd.DataFrame(results_b7)

=== Model Performance Comparison ===
Model 1 (km only) RMSE        : 17.2986
Model 2 (km + org OHE) RMSE   : 11.2901
Model 1 intercept & slope     : 44.2332 & 0.0758


,Regularization,RMSE,Total Coefficients,Near-Zero Coefficients
0,Ridge,11.289664,9,0
1,ElasticNet,11.291365,9,1
2,Lasso,11.294537,9,0


### Bài 8 & 9 - Pipeline Architecture and Cross-Validation Tuning

In [63]:
# --- Bài 8: Pipeline Explanation ---
# Fitting must strictly occur on training data to prevent DATA LEAKAGE.
# If transformations like StringIndexer categories or TF-IDF parameters are derived using both
# training and testing subsets, information from the testing target values implicitly influences
# model weights, generating over-optimistic evaluation metrics that fail in production.

# --- Bài 9: 3-Fold Cross Validator ---
# Configuration parameter grid contains 3 * 3 = 9 candidate hyperparameter combinations.

cv_lr_b9 = LinearRegression(featuresCol="features", labelCol="duration")

pipeline_b9 = Pipeline(stages=[
    StringIndexer(inputCol="org", outputCol="org_idx", handleInvalid="keep"),
    OneHotEncoder(inputCols=["org_idx"], outputCols=["org_dummy"]),
    VectorAssembler(inputCols=["km", "org_dummy"], outputCol="features"),
    cv_lr_b9
])

grid_b9 = (
    ParamGridBuilder()
    .addGrid(cv_lr_b9.regParam, [0.01, 0.1, 1.0])
    .addGrid(cv_lr_b9.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)

cv_b9 = CrossValidator(
    estimator=pipeline_b9,
    estimatorParamMaps=grid_b9,
    evaluator=RegressionEvaluator(labelCol="duration", metricName="rmse"),
    numFolds=3,
    seed=42,
    parallelism=2
)

# Perform split and fit CV
train_b9, test_b9 = reg_base_b6.randomSplit([0.8, 0.2], seed=42)
cv_model_b9 = cv_b9.fit(train_b9)

# Test set performance evaluation
cv_predictions = cv_model_b9.transform(test_b9)
test_rmse_b9 = RegressionEvaluator(labelCol="duration", metricName="rmse").evaluate(cv_predictions)

print("=== Cross-Validation Results ===")
print(f"Best configuration test set RMSE : {test_rmse_b9:.4f}")
print(f"Best regParam                    : {cv_model_b9.bestModel.stages[-1].getRegParam()}")
print(f"Best elasticNetParam             : {cv_model_b9.bestModel.stages[-1].getElasticNetParam()}")

=== Cross-Validation Results ===
Best configuration test set RMSE : 11.2302
Best regParam                    : 0.01
Best elasticNetParam             : 0.5


### Bài 10 - Classifier Ensemble Comparison

In [64]:
# Train and evaluate tree-based ensembles on the classified flight delays dataset
train_b10, test_b10 = flights_ready_b2.randomSplit([0.8, 0.2], seed=42)

ensemble_models = {
    "Decision Tree": DecisionTreeClassifier(featuresCol="features", labelCol="label", seed=42),
    "Random Forest": RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=30, seed=42),
    "Gradient-Boosted Trees (GBT)": GBTClassifier(featuresCol="features", labelCol="label", maxIter=20, seed=42)
}

comparison_b10 = []

for name, clf in ensemble_models.items():
    model_b10 = clf.fit(train_b10)
    predictions_b10 = model_b10.transform(test_b10)

    auc_val = BinaryClassificationEvaluator(labelCol="label").evaluate(predictions_b10)
    f1_val = MulticlassClassificationEvaluator(labelCol="label", metricName="f1").evaluate(predictions_b10)

    comparison_b10.append({
        "Model": name,
        "F1-Score": f1_val,
        "AUC": auc_val
    })

pd.DataFrame(comparison_b10).sort_values(by="AUC", ascending=False)

,Model,F1-Score,AUC
2,Gradient-Boosted Trees (GBT),0.664974,0.727417
1,Random Forest,0.631095,0.683176
0,Decision Tree,0.631709,0.598907


## 6. Câu hỏi ôn tập và yêu cầu nộp bài

Câu hỏi ôn tập

1. Regression và Classification khác nhau thế nào?
2. StringIndexer khác OneHotEncoder ở điểm nào?
3. Tại sao Spark ML cần cột features dạng vector?
4. Confusion matrix gồm những thành phần nào?
5. Khi nào Accuracy có thể gây hiểu nhầm?
6. TF-IDF giúp biểu diễn văn bản như thế nào?
7. RMSE và R² có ý nghĩa gì?
8. Lasso khác Ridge thế nào?
9. Data leakage xảy ra khi nào?
10. Pipeline hỗ trợ tránh leakage như thế nào?
11. Cross Validation dùng training set hay test set?
12. Random Forest khác Decision Tree đơn như thế nào?

**6. Câu hỏi ôn tập – Spark MLlib**

**1. Regression và Classification khác nhau thế nào?**

* Regression (hồi quy): Dự đoán giá trị số liên tục, ví dụ dự đoán giá nhà hoặc doanh thu.
* Classification (phân loại): Dự đoán nhãn hoặc nhóm, ví dụ phân loại chuyến bay thành đúng giờ và trễ chuyến.

**2. StringIndexer khác OneHotEncoder ở điểm nào?**

* StringIndexer: Chuyển nhãn dạng chuỗi thành chỉ số số học.
* OneHotEncoder: Chuyển các chỉ số phân loại thành vector nhị phân, giúp biểu diễn các nhóm mà không ngụ ý thứ tự hoặc khoảng cách số học giữa chúng.
Hai bước này thường được sử dụng liên tiếp trong quy trình tiền xử lý dữ liệu phân loại.

**3. Tại sao Spark ML cần cột features dạng vector?**
Spark ML sử dụng cột features để tập hợp các biến đầu vào thành một vector số học thống nhất. Nhờ đó, các thuật toán có thể tiếp nhận dữ liệu theo định dạng chuẩn, thuận tiện cho huấn luyện và dự đoán.

**4. Confusion matrix gồm những thành phần nào?**
Đối với bài toán phân loại nhị phân, confusion matrix gồm bốn thành phần:

* TP (True Positive): Dự đoán dương tính và thực tế dương tính.
* TN (True Negative): Dự đoán âm tính và thực tế âm tính.
* FP (False Positive): Dự đoán dương tính nhưng thực tế âm tính.
* FN (False Negative): Dự đoán âm tính nhưng thực tế dương tính.
Bốn thành phần này giúp tính Accuracy, Precision, Recall và F1-score để đánh giá mô hình từ nhiều góc độ.

**5. Khi nào Accuracy có thể gây hiểu nhầm?**
Accuracy có thể gây hiểu nhầm khi dữ liệu mất cân bằng giữa các lớp. Ví dụ, nếu 95% chuyến bay không bị trễ, mô hình luôn dự đoán “không trễ” vẫn đạt Accuracy 95%, dù không phát hiện được chuyến bay trễ nào.
Trong trường hợp này, cần xem xét thêm Precision, Recall, F1-score và confusion matrix để đánh giá khả năng nhận diện từng lớp.

**6. TF-IDF giúp biểu diễn văn bản như thế nào?**
TF-IDF (Term Frequency–Inverse Document Frequency) biểu diễn văn bản thành các trọng số số học dựa trên mức độ xuất hiện của từ trong một văn bản và mức độ phổ biến của từ trong toàn bộ tập tài liệu.

* TF: Đo mức độ xuất hiện của từ trong văn bản.
* IDF: Giảm trọng số của những từ xuất hiện phổ biến trong nhiều văn bản.
Nhờ đó, các từ có tính phân biệt cao đối với một văn bản có thể nhận trọng số lớn hơn, giúp mô hình học máy xử lý dữ liệu văn bản hiệu quả hơn.

**7. RMSE và R² có ý nghĩa gì?**

* RMSE (Root Mean Squared Error): Đo độ lớn sai số dự đoán theo đơn vị của biến mục tiêu. RMSE càng thấp thì sai số dự đoán thường càng nhỏ.
* R² (Coefficient of Determination): Đánh giá mức độ mô hình giải thích sự biến thiên của biến mục tiêu so với mô hình tham chiếu dựa trên giá trị trung bình.
R² gần 1 thường thể hiện khả năng giải thích tốt hơn. Tuy nhiên, R² có thể âm trên tập đánh giá nếu mô hình dự đoán kém hơn việc luôn dự đoán bằng giá trị trung bình. Cần đánh giá cả hai chỉ số trong bối cảnh dữ liệu cụ thể.

**8. Lasso khác Ridge thế nào?**
Cả hai đều là phương pháp hồi quy có điều chuẩn (regularization) nhằm hạn chế mô hình quá khớp bằng cách phạt các hệ số quá lớn.

* Lasso (L1): Có thể đưa một số hệ số về đúng 0, qua đó hỗ trợ lựa chọn biến.
* Ridge (L2): Thu nhỏ các hệ số về gần 0 nhưng thông thường không đưa chúng về đúng 0.
Lasso phù hợp khi muốn tạo mô hình thưa, còn Ridge hữu ích khi có nhiều biến tương quan với nhau. Việc lựa chọn còn phụ thuộc vào cấu trúc dữ liệu và mục tiêu phân tích.

**9. Data leakage xảy ra khi nào?**
Data leakage xảy ra khi thông tin không nên có tại thời điểm huấn luyện hoặc dự đoán bị đưa vào quy trình xây dựng mô hình, khiến kết quả đánh giá tốt một cách giả tạo.
Ví dụ:

* Chuẩn hóa dữ liệu trên toàn bộ tập dữ liệu trước khi chia train/test.
* Sử dụng thông tin chỉ xuất hiện sau thời điểm cần dự đoán làm biến đầu vào.
* Lựa chọn biến dựa trên cả tập test.
Data leakage khiến mô hình có thể hoạt động tốt trên dữ liệu đánh giá nhưng kém khi áp dụng vào thực tế.

**10. Pipeline hỗ trợ tránh leakage như thế nào?**
Pipeline trong Spark ML kết nối các bước tiền xử lý và huấn luyện thành một quy trình thống nhất. Khi kết hợp với việc chia dữ liệu đúng cách, Pipeline giúp các bước học tham số như StringIndexer, StandardScaler hoặc IDF được huấn luyện trên tập train thay vì toàn bộ dữ liệu.
Ví dụ, khi sử dụng Pipeline trong Cross Validation, các bước tiền xử lý được fit riêng trên tập huấn luyện của từng fold. Điều này giúp hạn chế việc thông tin từ tập validation lọt vào quá trình huấn luyện.
Tuy nhiên, Pipeline không tự động ngăn mọi loại leakage: người dùng vẫn phải chia dữ liệu đúng thời điểm, tránh tạo biến chứa thông tin tương lai và thiết kế quy trình đánh giá phù hợp.

**11. Cross Validation dùng training set hay test set?**
Cross Validation sử dụng training set, không sử dụng test set để lựa chọn mô hình hoặc siêu tham số.
Quy trình thông thường:

1. Chia dữ liệu thành training set và test set.
2. Thực hiện Cross Validation trên training set để lựa chọn mô hình và siêu tham số.
3. Huấn luyện mô hình cuối cùng trên training set với cấu hình đã chọn.
4. Đánh giá một lần trên test set để ước lượng khả năng tổng quát hóa.
Test set cần được giữ độc lập trong quá trình lựa chọn mô hình để tránh đánh giá quá lạc quan.

**12. Random Forest khác Decision Tree đơn như thế nào?**

* Decision Tree: Xây dựng một cây quyết định duy nhất bằng cách chia dữ liệu theo các điều kiện. Mô hình dễ diễn giải nhưng có thể nhạy cảm với biến động của dữ liệu huấn luyện.
* Random Forest: Kết hợp nhiều cây được huấn luyện trên các mẫu bootstrap và các tập biến được lựa chọn ngẫu nhiên. Kết quả dự đoán được tổng hợp từ nhiều cây.
Random Forest thường ổn định hơn và giảm phương sai so với một cây đơn, nhưng có thể tốn tài nguyên hơn và khó diễn giải trực tiếp hơn.